# 🎨 Example 4: Coloured Globe (Complete Pipeline)

Welcome to the ultimate `globe3d` walkthrough! In this tutorial, we will combine every single feature of the library into a single model: hollowing, topography, coastlines, magnet slots, and independent inner/outer surface coloring. 

### 🌎 Scientific & Design Context
- **Tomography Color Mapping**: In this example, we visualize deep-Earth structures at the Core-Mantle Boundary (CMB, 2,850 km depth) from the seismic tomography model SP12RTS. Vertices over low-velocity zones (hot, buoyant mantle upwellings like the Pacific plume) will be colored red, while high-velocity zones (cold, dense, subducted slabs) will be colored blue.
- **Independent Cavity Painting**: Since the model is hollow and split in half, we color the inner cavity walls a clean, neutral gray. This prevents the outer datasets from bleeding inside, ensuring a professional, readable model.

## Step 1: Import Libraries

We import `globe3d`'s object-oriented components along with standard helper libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from globe3d import (
    GlobeModel,
    GeographicGrid,
    GridDisplacer,
    LineDisplacer,
    GridColourer,
    ConstantColourer,
    calculate_displacement_scale
)

## Step 2: Generate a Hollow Sphere

We generate an 8,000-point outer shell, and set an `inner_ratio` of `0.5` which automatically creates the inner cavity wall.

In [ ]:
model_radius_mm = 40.0

model = GlobeModel(
    n_points=8000,
    radius=model_radius_mm,
    hollow=True,
    inner_ratio=0.5,
)
print(f"Outer shell: {model.outer.vertices.shape[0]} vertices")
print(f"Inner cavity: {model.inner.vertices.shape[0]} vertices")

## Step 3: Apply Topography and Coastline Step

We load and downsample ETOPO topography, apply it with a 40× exaggeration, and overlay a sharp 0.8 mm coastline step using our vector shapefile.

In [ ]:
# Load and downsample topography
topo_grid = GeographicGrid.from_netcdf(
    "../inputs/ETOPO_2022_v1_60s_N90W180_surface.nc", 'lat', 'lon', 'z'
)
topo_grid_ds = GeographicGrid(
    lats=topo_grid.lats[::10],
    lons=topo_grid.lons[::10],
    grid=topo_grid.grid[::10, ::10]
)

# Scale and displace
topo_units = 'm'
scale = calculate_displacement_scale(
    model_radius_mm, vertical_exagg=40.0, grid_units=topo_units
)
model.outer.displace(GridDisplacer(topo_grid_ds), scale=scale)

# Apply coastline step ribbon
model.outer.displace(LineDisplacer(
    shapefile_path="../inputs/coastlines/ne_110m_coastline.shp",
    displacement=0.8,
    width_degrees=0.5
))

## Step 4: Configure Magnets

We configure 3 magnet pockets per hemisphere with reinforcing support bosses.

In [ ]:
model.configure_magnets(
    diameter=5.0,
    height=2.0,
    horizontal_tolerance=0.15,
    vertical_tolerance=0.10,
    vertical_offset=0.20,
    min_thickness=1.5,
    n_magnets=3,
    add_bosses=True,
)

## Step 5: Apply Independent Surface Coloring

We color the outward-facing (outer shell) surfaces based on the deep mantle seismic tomography model, and paint the inward-facing cavity walls to a solid gray.

In [ ]:
# Load seismic tomography slice
tomo_grid = GeographicGrid.from_netcdf(
    "../inputs/s40_depth_slice_2850.grd", 'y', 'x', 'z'
)

# 1. Color outer surfaces using a Red-to-Blue colormap
model.outer.colour(
    GridColourer(tomo_grid, colormap='RdBu_r', vmin=-2.0, vmax=2.0),
    selection='outward_facing',
)

# 2. Paint interior cavity to solid gray
model.outer.colour(
    ConstantColourer([0.6, 0.6, 0.6]),
    selection='inward_facing',
)

## Step 6: Preview Colored Hemispheres in 3D

Let's split the model and preview the colored vertices to verify that the outer details and inner cavity painting are correct.

In [ ]:
# Generate hemispheres for preview
top_half, bottom_half = model.generate_hemispheres(engine='manifold')

fig = plt.figure(figsize=(12, 6))

top_colors = top_half.visual.vertex_colors[:, :3].astype(float) / 255.0
bottom_colors = bottom_half.visual.vertex_colors[:, :3].astype(float) / 255.0

ax1 = fig.add_subplot(121, projection='3d')
pts_top = top_half.vertices
ax1.scatter(pts_top[:, 0], pts_top[:, 1], pts_top[:, 2], c=top_colors, s=2)
ax1.set_title("Colored Top Hemisphere")

ax2 = fig.add_subplot(122, projection='3d')
pts_bot = bottom_half.vertices
ax2.scatter(pts_bot[:, 0], pts_bot[:, 1], pts_bot[:, 2], c=bottom_colors, s=2)
ax2.set_title("Colored Bottom Hemisphere")

plt.show()

## Step 7: Export to OBJ with Vertex Colors

We export both hemispheres to OBJ files, ready to load into Bambu Studio or OrcaSlicer for multi-color printing.

In [ ]:
model.export_hemispheres(
    "../outputs/example_4_top.obj",
    "../outputs/example_4_bottom.obj",
    engine='manifold',
)
print("Colored OBJ hemispheres exported successfully!")

## 8. Preview the Model in 3D

Preview the interactive 3D model:

In [ ]:
model.preview()